In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.ipa_processing import syllabify_ipa
SEP = "."

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri75.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [3]:
df

,syllable1_start,syllable1_end,syllable2_start,syllable2_end,syllable1,syllable2,context,word,word_id,art_f,ac_f,subject
0,5.190000,5.430000,5.430000,5.580000,k_l_oʊ,s_l_i,k_l_oʊ,closely,17.0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,5.840000,6.090000,6.090000,6.160000,t_ɹ_æ,v_ʌ,t_ɹ_æ,traveler,20.0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,6.090000,6.160000,6.160000,6.400000,v_ʌ,l_ɚ,t_ɹ_æ.v_ʌ,traveler,20.0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,7.080954,7.152108,7.152108,7.421014,ɚ,aʊ_n_d,ɚ,around,24.0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,10.148720,10.225550,10.225550,10.674285,ʌ,t_ɛ_m_p_t,ʌ,attempt,35.0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...,...,...,...,...,...,...
3041,49.250000,49.840000,49.840000,50.130000,b_ʌ,n_æ,b_ʌ,banana,125.0,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
3042,49.840000,50.130000,50.130000,50.230000,n_æ,n_ʌ,b_ʌ.n_æ,banana,125.0,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
3043,50.530000,50.770000,50.770000,50.890000,g_ɹ_æ_n_d,f_ɑ,g_ɹ_æ_n_d,grandfather,127.0,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
3044,50.770000,50.890000,50.890000,51.060000,f_ɑ,ð_ɚ,g_ɹ_æ_n_d.f_ɑ,grandfather,127.0,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [4]:
with open("../../../phone_based/pronunciation_counts/pronunciation_counts_ae.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="ae"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for syllable1, syllable2 in tqdm.tqdm(zip(df["syllable1"], df["syllable2"]), total=len(df)):
    bigrams.add(syllable1+SEP+syllable2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    syllables = pronunciation.split(SEP)
    for syllable in syllables:
        if syllable not in monogram_counts:
            monogram_counts[syllable] = 0
        monogram_counts[syllable] += count

100%|██████████| 3046/3046 [00:00<00:00, 896739.66it/s]


100%|██████████| 48576/48576 [00:00<00:00, 1601812.27it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 48576/48576 [00:00<00:00, 313028.15it/s]


In [7]:
df["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["syllable1"] + SEP + x["syllable2"]), axis=1)
df["syllable2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1)